In [1]:
import pandas as pd

file_path = '/content/drive/MyDrive/Mlops/BankNote_Authentication.csv'
df = pd.read_csv(file_path)
display(df.head())

,variance,skewness,curtosis,entropy,class
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


In [3]:
X = df.drop('class', axis=1)
y = df['class']

print('X head:')
display(X.head())

print('\ny head:')
display(y.head())

X head:


,variance,skewness,curtosis,entropy
0,3.62160,8.6661,-2.8073,-0.44699
1,4.54590,8.1674,-2.4586,-1.46210
2,3.86600,-2.6383,1.9242,0.10645
3,3.45660,9.5228,-4.0112,-3.59440
4,0.32924,-4.4552,4.5718,-0.98880



y head:


,class
0,0
1,0
2,0
3,0
4,0


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print('Scaled X head:')
display(X_scaled_df.head())

Scaled X head:


,variance,skewness,curtosis,entropy
0,1.121806,1.149455,-0.975970,0.354561
1,1.447066,1.064453,-0.895036,-0.128767
2,1.207810,-0.777352,0.122218,0.618073
3,1.063742,1.295478,-1.255397,-1.144029
4,-0.036772,-1.087038,0.736730,0.096587


In [6]:
from sklearn.model_selection import train_test_split

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.2, random_state=42)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

X_train shape: (1097, 4)
X_test shape: (275, 4)
y_train shape: (1097,)
y_test shape: (275,)


In [10]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

print('Random Forest model trained successfully!')

Random Forest model trained successfully!


In [25]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

accuracy = int(accuracy_score(y_test, y_pred)*100)


print(f'Model Accuracy: {accuracy}%')

Model Accuracy: 99%


In [23]:
import pickle

model_filename = 'random_forest_model.pkl'

with open(model_filename, 'wb') as file:
    pickle.dump(model, file)


In [26]:
%%writefile /content/drive/MyDrive/Mlops/app.py

import streamlit as st
import pickle
import pandas as pd
from sklearn.preprocessing import StandardScaler
import os

# Define the paths for the model and the data
MODEL_PATH = '/content/drive/MyDrive/Mlops/random_forest_model.pkl'
DATA_PATH = '/content/drive/MyDrive/Mlops/BankNote_Authentication.csv'

# --- Load the trained model ---
# Check if the model file exists
if not os.path.exists(MODEL_PATH):
    st.error(f"Model file not found at: {MODEL_PATH}")
    st.stop()

with open(MODEL_PATH, 'rb') as file:
    model = pickle.load(file)

# --- Load data and fit scaler ---
# This is necessary because the model was trained on scaled data.
# We need the same scaler to preprocess new input data.
if not os.path.exists(DATA_PATH):
    st.error(f"Data file not found at: {DATA_PATH}")
    st.stop()

df_original = pd.read_csv(DATA_PATH)
X_for_scaler = df_original.drop('class', axis=1)

scaler = StandardScaler()
scaler.fit(X_for_scaler)

# --- Streamlit UI ---
st.title('Bank Note Authentication Prediction')
st.write('Enter the values for the bank note features to get a prediction:')

# Input fields for features
variance = st.number_input('Variance', value=0.0)
skewness = st.number_input('Skewness', value=0.0)
curtosis = st.number_input('Curtosis', value=0.0)
entropy = st.number_input('Entropy', value=0.0)

# Prediction button
if st.button('Predict'):
    # Create a DataFrame from input values
    input_data = pd.DataFrame([{
        'variance': variance,
        'skewness': skewness,
        'curtosis': curtosis,
        'entropy': entropy
    }])

    # Scale the input data using the fitted scaler
    scaled_input_data = scaler.transform(input_data)

    # Make prediction
    prediction = model.predict(scaled_input_data)
    prediction_proba = model.predict_proba(scaled_input_data)

    # Display prediction
    st.subheader('Prediction:')
    if prediction[0] == 0:
        st.success(f"The bank note is likely Genuine. (Probability: {prediction_proba[0][0]:.2f})")
    else:
        st.error(f"The bank note is likely Forged. (Probability: {prediction_proba[0][1]:.2f})")

st.write("--- Developed with Streamlit and Random Forest Model ---")


Overwriting /content/drive/MyDrive/Mlops/app.py
